In [ ]:
import os
from pathlib import Path

# Force a stable W&B directory across sessions/devices
WANDB_DIR = Path("/home/MohammadNabulsi/Essay Evaluator/experiments/artifacts/wandb")
os.environ["WANDB_DIR"] = str(WANDB_DIR)
WANDB_DIR.mkdir(parents=True, exist_ok=True)


# Qwen3 Epoch Adapter Evaluation (No Training)

This notebook is evaluation-only:
- Preload base model once.
- Load/switch adapters for epoch 1, 2, 3.
- Compute score metrics: precision, recall, F1.
- Compute rationale metrics: ROUGE, BLEU, BERTScore.
- Log generation progress at ~10% increments.


In [ ]:
import gc
import json
import random
import re
from pathlib import Path
from typing import Any, Dict, List, Optional

import evaluate
import numpy as np
import pandas as pd
import torch
from peft import PeftModel
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from unsloth import FastLanguageModel


In [ ]:
SEED = 3407
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

OUTPUT_ROOT = Path("../artifacts/abstract_evaluator_qwen3_sft")
DATA_PATH = Path("../../data/processed/final_sft_dataset_1220.jsonl")
MODEL_NAME = "Qwen/Qwen3-8B"
RUN_NAME = "qwen3_8b_abstract_evaluator_lora_no_quant"
MAX_SEQ_LENGTH = 2048

TRAIN_ROWS, VAL_ROWS, TEST_ROWS = 910, 100, 100
MAX_NEW_TOKENS = 80
BATCH_SIZE = 8

EPOCH_ROOT = OUTPUT_ROOT / "models" / RUN_NAME / "epoch_adapters"
ADAPTERS = {
    "epoch_1": EPOCH_ROOT / "epoch_1",
    "epoch_2": EPOCH_ROOT / "epoch_2",
    "epoch_3": EPOCH_ROOT / "epoch_3",
}

for k, v in ADAPTERS.items():
    print(k, v, "exists=", v.exists())


In [ ]:
REQUIRED_COLUMNS = ["paper_id", "submission", "score", "rationale"]
DEFAULT_TASK = "Evaluate the quality of the following research abstract for conference acceptance."
DEFAULT_REFERENCE = "A strong research abstract clearly presents the problem, methodology, contribution, and experimental evidence."
DEFAULT_RUBRIC = {
    "score_scale": {
        "0": "Very poor abstract: missing most core components, unclear, generic, or unusable.",
        "1": "Weak abstract: contains a few useful elements but major components are missing or vague.",
        "2": "Borderline abstract: understandable but incomplete; some important components are weak or missing.",
        "3": "Good abstract: mostly complete, clear, and logically structured, with minor weaknesses.",
        "4": "Excellent abstract: complete, clear, concise, well-structured, and strongly communicates the paper's contribution and evidence.",
    },
    "criteria": {
        "1_background_or_context": "Provides enough background or introduction to understand the research area and motivation.",
        "2_problem_statement": "Clearly identifies the research problem, gap, or limitation being addressed.",
        "3_objective_or_purpose": "States the main objective, research question, or purpose of the work.",
        "4_methodology": "Explains the methods, approach, model, experiment, dataset, or procedure used.",
        "5_results_or_findings": "Reports concrete results, findings, observations, or evidence rather than only intentions.",
        "6_contribution": "Clarifies what is new, useful, or significant about the work.",
        "7_conclusion_or_implication": "Provides a conclusion, implication, impact, or takeaway from the work.",
        "8_clarity_and_conciseness": "Uses clear, precise, and concise language without unnecessary vagueness or filler.",
        "9_logical_flow": "Presents the abstract in a coherent order: context/problem -> objective -> method -> results -> contribution.",
        "10_specificity_and_precision": "Includes specific details (e.g., methods, data, outcomes) rather than broad generic claims.",
    },
}


def load_final_dataframe(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"DATA_PATH does not exist: {path}")
    if path.suffix.lower() == ".jsonl":
        return pd.read_json(path, lines=True)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    return pd.read_json(path)


def format_rubric(rubric: Dict[str, Any]) -> str:
    if isinstance(rubric, dict) and "score_scale" in rubric and "criteria" in rubric:
        score_lines = ["Score scale:"]
        for k, v in sorted(rubric["score_scale"].items(), key=lambda x: int(str(x[0]))):
            score_lines.append(f"{k} = {v}")
        crit_lines = ["Criteria:"]
        for k, v in sorted(rubric["criteria"].items(), key=lambda x: str(x[0])):
            crit_lines.append(f"{k}: {v}")
        return "\n".join(score_lines + [""] + crit_lines)
    return "\n".join([f"{k} = {v}" for k, v in sorted(rubric.items(), key=lambda x: str(x[0]))])


def make_user_prompt(row: pd.Series) -> str:
    title_block = ""
    if "title" in row and pd.notna(row["title"]) and str(row["title"]).strip():
        title_block = f"\nTitle:\n{str(row['title']).strip()}\n"
    return (
        "/no_think\n"
        f"Task:\n{row['task']}\n\n"
        f"Reference:\n{row['reference']}\n\n"
        f"Rubric:\n{format_rubric(row['rubric'])}\n"
        f"{title_block}\n"
        f"Submission:\n{row['submission']}\n\n"
        "Return only valid JSON with exactly these keys: score, rationale.\n"
        "Do not include markdown, analysis, or extra text."
    )


def make_messages(row: pd.Series) -> List[Dict[str, str]]:
    return [
        {"role": "system", "content": "You are a strict research abstract evaluator. You return only valid JSON."},
        {"role": "user", "content": make_user_prompt(row)},
        {"role": "assistant", "content": json.dumps({"score": int(row["score"]), "rationale": str(row["rationale"]).strip()}, ensure_ascii=False)},
    ]


In [ ]:
def _pick_degradation_col(df: pd.DataFrame) -> Optional[str]:
    for c in ["degradation_type", "degradation", "degrade_type", "perturbation_type"]:
        if c in df.columns:
            return c
    return None


def _build_joint_strata(df: pd.DataFrame, score_col: str = "score", deg_col: Optional[str] = None) -> pd.Series:
    score_part = df[score_col].astype(str)
    if deg_col is None:
        return score_part
    deg_part = df[deg_col].fillna("missing").astype(str)
    joint = score_part + "||" + deg_part
    counts = joint.value_counts()
    rare = counts[counts < 2].index
    if len(rare) > 0:
        joint = joint.where(~joint.isin(rare), score_part + "||__other__")
    if joint.value_counts().min() < 2:
        return score_part
    return joint


def split_fixed_counts_stratified(df: pd.DataFrame, train_rows=TRAIN_ROWS, val_rows=VAL_ROWS, test_rows=TEST_ROWS, seed=SEED):
    total_needed = train_rows + val_rows + test_rows
    if len(df) < total_needed:
        raise ValueError(f"Need at least {total_needed} rows, found {len(df)}")
    base = df.sample(frac=1.0, random_state=seed).head(total_needed).reset_index(drop=True)
    deg_col = _pick_degradation_col(base)
    strata_all = _build_joint_strata(base, score_col="score", deg_col=deg_col)

    try:
        rest_df, test_df = train_test_split(base, test_size=test_rows, random_state=seed, stratify=strata_all)
    except ValueError:
        rest_df, test_df = train_test_split(base, test_size=test_rows, random_state=seed, stratify=base["score"])

    try:
        strata_rest = _build_joint_strata(rest_df, score_col="score", deg_col=deg_col if deg_col in rest_df.columns else None)
        train_df, val_df = train_test_split(rest_df, test_size=val_rows, random_state=seed, stratify=strata_rest)
    except ValueError:
        train_df, val_df = train_test_split(rest_df, test_size=val_rows, random_state=seed, stratify=rest_df["score"])

    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)


df = load_final_dataframe(DATA_PATH)
for c in REQUIRED_COLUMNS:
    if c not in df.columns:
        raise ValueError(f"Missing required column: {c}")
if "task" not in df.columns:
    df["task"] = DEFAULT_TASK
if "reference" not in df.columns:
    df["reference"] = DEFAULT_REFERENCE
if "rubric" not in df.columns:
    df["rubric"] = [DEFAULT_RUBRIC] * len(df)

train_df, val_df, test_df = split_fixed_counts_stratified(df)
for part in [train_df, val_df, test_df]:
    part["messages"] = part.apply(make_messages, axis=1)

print("Split sizes:", len(train_df), len(val_df), len(test_df))


In [ ]:
def _normalize_messages_for_template(messages):
    if isinstance(messages, dict):
        roles = messages.get("role", [])
        contents = messages.get("content", [])
        if isinstance(roles, list) and isinstance(contents, list):
            return [{"role": str(r), "content": str(c)} for r, c in zip(roles, contents)]
    if isinstance(messages, list):
        out = []
        for m in messages:
            if isinstance(m, dict):
                out.append({"role": str(m.get("role", "user")), "content": str(m.get("content", ""))})
        return out
    return [{"role": "user", "content": str(messages)}]


def apply_qwen3_chat_template(tokenizer, messages, add_generation_prompt=False):
    messages = _normalize_messages_for_template(messages)
    try:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=add_generation_prompt, enable_thinking=False)
    except TypeError:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=add_generation_prompt)


def extract_json_object(text: str) -> Optional[Dict[str, Any]]:
    if not isinstance(text, str):
        return None
    try:
        obj = json.loads(text)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass
    match = re.search(r"\{.*?\}", text, flags=re.S)
    if match:
        try:
            obj = json.loads(match.group(0))
            if isinstance(obj, dict):
                return obj
        except Exception:
            return None
    return None


def parse_score(text: str) -> Optional[int]:
    obj = extract_json_object(text)
    if obj is not None and "score" in obj:
        try:
            s = int(obj["score"])
            if 0 <= s <= 4:
                return s
        except Exception:
            pass
    match = re.search(r'"?score"?\s*[:=]\s*([0-4])', str(text))
    if match:
        return int(match.group(1))
    return None


def parse_rationale(text: str) -> str:
    obj = extract_json_object(text)
    if obj is not None and "rationale" in obj:
        rationale = str(obj["rationale"]).strip()
        if rationale:
            return rationale

    fallback = str(text).strip()
    return fallback if fallback else "__EMPTY_OUTPUT__"


def make_inference_prompt(tokenizer, messages):
    system_user = [m for m in messages if m["role"] in ["system", "user"]]
    return apply_qwen3_chat_template(tokenizer, system_user, add_generation_prompt=True)


In [ ]:
# 1) Load base once
base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=torch.bfloat16,
    load_in_4bit=True,
)

tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

FastLanguageModel.for_inference(base_model)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id
base_model.generation_config.eos_token_id = tokenizer.eos_token_id

# 2) Adapter switching
peft_model = None
loaded_adapters = set()


def get_model_for_adapter(adapter_dir: Path, adapter_name: str):
    global peft_model, loaded_adapters
    if peft_model is None:
        peft_model = PeftModel.from_pretrained(base_model, str(adapter_dir), adapter_name=adapter_name, is_trainable=False)
        loaded_adapters.add(adapter_name)
    else:
        if adapter_name not in loaded_adapters:
            peft_model.load_adapter(str(adapter_dir), adapter_name=adapter_name, is_trainable=False)
            loaded_adapters.add(adapter_name)
        peft_model.set_adapter(adapter_name)

    FastLanguageModel.for_inference(peft_model)
    peft_model.generation_config.pad_token_id = tokenizer.pad_token_id
    peft_model.generation_config.eos_token_id = tokenizer.eos_token_id
    return peft_model

print("Base preloaded. Ready to evaluate adapters.")


In [ ]:
@torch.no_grad()
def generate_predictions(eval_df: pd.DataFrame, model, tokenizer, max_new_tokens=MAX_NEW_TOKENS, batch_size=BATCH_SIZE):
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    prompts = [make_inference_prompt(tokenizer, msgs) for msgs in eval_df["messages"]]
    total = len(prompts)
    total_batches = (total + batch_size - 1) // batch_size

    predictions = []
    pred_token_lens = []
    device = "cuda" if torch.cuda.is_available() else "cpu"
    right_padding_batches = 0

    log_marks = set()
    for pct in range(10, 101, 10):
        b = max(1, int(np.ceil(total_batches * pct / 100.0)))
        log_marks.add(b)

    for batch_idx, start in enumerate(range(0, total, batch_size), start=1):
        batch_prompts = prompts[start:start + batch_size]

        # Unsloth can flip this during generation; force left-padding every batch.
        tokenizer.padding_side = "left"

        inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_SEQ_LENGTH,
        ).to(device)

        last_is_pad = int((inputs["input_ids"][:, -1] == tokenizer.pad_token_id).sum().item())
        if last_is_pad > 0:
            right_padding_batches += 1

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            top_p=1.0,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

        prompt_len = inputs["input_ids"].shape[1]
        generated = outputs[:, prompt_len:]
        texts = tokenizer.batch_decode(generated, skip_special_tokens=True)

        predictions.extend([str(t).strip() for t in texts])
        pred_token_lens.extend([int(t.numel()) for t in generated])

        if batch_idx in log_marks:
            print(f"Progress: {int(round(batch_idx / total_batches * 100))}% ({batch_idx}/{total_batches} batches)")

    out = eval_df.copy().reset_index(drop=True)
    out["prediction_text"] = predictions
    out["pred_tokens"] = pred_token_lens
    out["pred_score"] = out["prediction_text"].apply(parse_score)
    out["pred_rationale"] = out["prediction_text"].apply(parse_rationale)
    out["pred_rationale_char_len"] = out["pred_rationale"].astype(str).str.len()
    pred_text_stripped = out["prediction_text"].astype(str).str.strip()
    raw_empty_pred_count = int((pred_text_stripped == "").sum())
    placeholder_count = int((out["pred_rationale"] == "__EMPTY_OUTPUT__").sum())
    placeholder_from_nonempty_raw_count = int(((out["pred_rationale"] == "__EMPTY_OUTPUT__") & (pred_text_stripped != "")).sum())

    print(f"Right-padding-warning-condition batches: {right_padding_batches}")
    print(f"Raw model-empty prediction_text rows: {raw_empty_pred_count}")
    print(f"Pred_rationale == __EMPTY_OUTPUT__ rows: {placeholder_count}")
    print(f"Pred_rationale == __EMPTY_OUTPUT__ with NON-empty raw prediction_text rows: {placeholder_from_nonempty_raw_count}")

    return out


In [ ]:
rouge_metric = evaluate.load("rouge")
bleu_metric = evaluate.load("bleu")
bertscore_metric = evaluate.load("bertscore")


def compute_metrics(pred_df: pd.DataFrame) -> Dict[str, float]:
    out = {}
    valid = pred_df["pred_score"].notna()
    out["json_parse_rate"] = float(valid.mean())

    if valid.any():
        y_true = pred_df.loc[valid, "score"].astype(int).values
        y_pred = pred_df.loc[valid, "pred_score"].astype(int).values

        p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
        p_weight, r_weight, f1_weight, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)

        out["score_accuracy"] = float(accuracy_score(y_true, y_pred))
        out["score_precision_macro"] = float(p_macro)
        out["score_recall_macro"] = float(r_macro)
        out["score_f1_macro"] = float(f1_macro)
        out["score_precision_weighted"] = float(p_weight)
        out["score_recall_weighted"] = float(r_weight)
        out["score_f1_weighted"] = float(f1_weight)
        out["score_mae"] = float(np.abs(y_true - y_pred).mean())
    else:
        out["score_accuracy"] = 0.0
        out["score_precision_macro"] = 0.0
        out["score_recall_macro"] = 0.0
        out["score_f1_macro"] = 0.0
        out["score_precision_weighted"] = 0.0
        out["score_recall_weighted"] = 0.0
        out["score_f1_weighted"] = 0.0
        out["score_mae"] = float("nan")

    preds = pred_df["pred_rationale"].fillna("").astype(str).tolist()
    refs = pred_df["rationale"].fillna("").astype(str).tolist()

    rouge = rouge_metric.compute(predictions=preds, references=refs)
    out.update({f"rouge_{k}": float(v) for k, v in rouge.items()})

    bleu = bleu_metric.compute(predictions=preds, references=[[r] for r in refs])
    out["bleu"] = float(bleu["bleu"])

    bert = bertscore_metric.compute(
        predictions=preds,
        references=refs,
        lang="en",
    )
    out["bertscore_precision"] = float(np.mean(bert["precision"]))
    out["bertscore_recall"] = float(np.mean(bert["recall"]))
    out["bertscore_f1"] = float(np.mean(bert["f1"]))

    out["mean_pred_tokens"] = float(pred_df["pred_tokens"].mean())
    out["p95_pred_tokens"] = float(pred_df["pred_tokens"].quantile(0.95))
    out["max_pred_tokens"] = float(pred_df["pred_tokens"].max())
    out["token_limit_hit_rate"] = float((pred_df["pred_tokens"] >= MAX_NEW_TOKENS).mean())
    return out


In [ ]:
eval_out_dir = OUTPUT_ROOT / "eval" / RUN_NAME / "adapter_eval_preload_metrics"
eval_out_dir.mkdir(parents=True, exist_ok=True)

all_metrics = []

for tag, adapter_dir in ADAPTERS.items():
    if not adapter_dir.exists():
        print("Skipping missing adapter:", adapter_dir)
        continue

    print(f"\n===== Evaluating {tag} =====")
    model = get_model_for_adapter(adapter_dir, adapter_name=tag)

    pred_df = generate_predictions(
        test_df,
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=MAX_NEW_TOKENS,
        batch_size=BATCH_SIZE,
    )

    pred_df.to_csv(eval_out_dir / f"test_predictions_{tag}.csv", index=False)

    m = compute_metrics(pred_df)
    row = {
        "checkpoint_tag": tag,
        "max_new_tokens": MAX_NEW_TOKENS,
        **{f"test/{k}": v for k, v in m.items()},
    }

    with open(eval_out_dir / f"metrics_{tag}.json", "w", encoding="utf-8") as f:
        json.dump(row, f, indent=2, ensure_ascii=False)

    all_metrics.append(row)

metrics_df = pd.DataFrame(all_metrics)
metrics_df.to_csv(eval_out_dir / "all_checkpoint_metrics.csv", index=False)
metrics_df


In [ ]:
cols = [
    "checkpoint_tag",
    "max_new_tokens",
    "test/json_parse_rate",
    "test/score_precision_macro",
    "test/score_recall_macro",
    "test/score_f1_macro",
    "test/score_precision_weighted",
    "test/score_recall_weighted",
    "test/score_f1_weighted",
    "test/rouge_rouge1",
    "test/rouge_rougeL",
    "test/bleu",
    "test/bertscore_precision",
    "test/bertscore_recall",
    "test/bertscore_f1",
    "test/mean_pred_tokens",
    "test/p95_pred_tokens",
    "test/max_pred_tokens",
    "test/token_limit_hit_rate",
]
metrics_df[cols]


## Interpreting Rationale Length vs `max_new_tokens`

`max_new_tokens` is only an upper bound. Rationale lengths do not have to equal it.
Use `token_limit_hit_rate` and `max_pred_tokens` to detect clipping.
- If `token_limit_hit_rate` is high, increase `MAX_NEW_TOKENS`.
- If low, `80` is likely sufficient.
